In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_web_traffic
from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()



Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [2]:
from src.features import add_time_features
DATA_ROOT = project_root / "data" / "datathon-2026-round-1"
df = load_web_traffic(data_root=DATA_ROOT)
drops_cols = ["bounce_rate", "traffic_source"]
df.drop(columns=drops_cols, inplace=True)
df = add_time_features(df, date_col="date")
df = df[df["year"] != 2012].reset_index(drop=True)

In [3]:
import pandas as pd
from src.models import SklearnRegressorConfig
from src.pipelines import train_validate_models

model_df = df.sort_values("date").reset_index(drop=True)

time_feature_cols = [
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "is_month_end",
    "is_month_start",
    "is_weekend",
]

traffic_feature_cols = [
    c
    for c in model_df.select_dtypes(include="number").columns
    if c not in time_feature_cols
]

# targets are no longer mapping, just a list of column names
targets = traffic_feature_cols
model_config = SklearnRegressorConfig(
    model_type="random_forest",
    random_state=42,
    n_estimators=300,
    max_depth=None,
)

results = train_validate_models(
    df=model_df,
    features=time_feature_cols,
    targets=targets,
    model_config=model_config,
    date_col="date",
    valid_fraction=0.2,
)

train_df = results["train_df"]
valid_df = results["valid_df"]
predictions = results["predictions"]
metrics = results["metrics"]

datasets_by_target = {}
for target in targets:
    datasets_by_target[target] = {
        "train": train_df[["date", *time_feature_cols, target]].copy(),
        "valid": valid_df[["date", *time_feature_cols, target]].copy(),
        "X_train": train_df[time_feature_cols].copy(),
        "X_valid": valid_df[time_feature_cols].copy(),
        "y_train": train_df[target].copy(),
        "y_valid": valid_df[target].copy(),
    }

pred_wide_df = pd.DataFrame({"date": valid_df["date"].to_numpy()})
for target, y_pred in predictions.items():
    pred_wide_df[target] = y_pred.to_numpy()

metrics_df = (
    pd.DataFrame(metrics).T.reset_index().rename(columns={"index": "target_feature"})
    .sort_values("mae", ascending=True)
    .reset_index(drop=True)
)

pred_long_df = pred_wide_df.melt(
    id_vars="date",
    var_name="predicted_feature",
    value_name="predicted_value",
)

In [4]:
metrics_df

,target_feature,mae,rmse,mape,smape,r2
0,avg_session_duration_sec,56.295832,66.524558,31.542072,27.590354,-0.074822
1,unique_visitors,2042.347405,2704.457050,8.718895,9.011691,0.884095
2,sessions,2407.489270,3198.850805,7.885880,8.182820,0.902825
3,page_views,20488.520971,26206.202136,15.658496,15.950748,0.711855


In [5]:
import pandas as pd
from src.models import SklearnRegressorConfig
from src.pipelines import train_validate_models

model_df_xgb = df.sort_values("date").reset_index(drop=True)

time_feature_cols_xgb = [
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "is_month_end",
    "is_month_start",
    "is_weekend",
]

traffic_feature_cols_xgb = [
    c
    for c in model_df_xgb.select_dtypes(include="number").columns
    if c not in time_feature_cols_xgb
]

targets_xgb = traffic_feature_cols_xgb

model_config_xgb = SklearnRegressorConfig(
    model_type="xgboost",
    random_state=42,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
)

results_xgb = train_validate_models(
    df=model_df_xgb,
    features=time_feature_cols_xgb,
    targets=targets_xgb,
    model_config=model_config_xgb,
    date_col="date",
    valid_fraction=0.2,
)

train_df_xgb = results_xgb["train_df"]
valid_df_xgb = results_xgb["valid_df"]
predictions_xgb = results_xgb["predictions"]
metrics_xgb = results_xgb["metrics"]

datasets_by_target_xgb = {}
for target in targets_xgb:
    datasets_by_target_xgb[target] = {
        "train": train_df_xgb[["date", *time_feature_cols_xgb, target]].copy(),
        "valid": valid_df_xgb[["date", *time_feature_cols_xgb, target]].copy(),
        "X_train": train_df_xgb[time_feature_cols_xgb].copy(),
        "X_valid": valid_df_xgb[time_feature_cols_xgb].copy(),
        "y_train": train_df_xgb[target].copy(),
        "y_valid": valid_df_xgb[target].copy(),
    }

pred_wide_df_xgb = pd.DataFrame({"date": valid_df_xgb["date"].to_numpy()})
for target, y_pred in predictions_xgb.items():
    pred_wide_df_xgb[target] = y_pred.to_numpy()

metrics_df_xgb = (
    pd.DataFrame(metrics_xgb).T.reset_index().rename(columns={"index": "target_feature"})
    .sort_values("mae", ascending=True)
    .reset_index(drop=True)
)

pred_long_df_xgb = pred_wide_df_xgb.melt(
    id_vars="date",
    var_name="predicted_feature",
    value_name="predicted_value",
)

pred_wide_df_xgb.head(), pred_long_df_xgb.head()

(        date      sessions  unique_visitors    page_views  \
 0 2020-12-31  19757.783203     15452.872070  80074.078125   
 1 2021-01-01  14473.414062     12301.527344  55172.414062   
 2 2021-01-02  14265.412109     12779.334961  62983.476562   
 3 2021-01-03  14742.094727     12383.919922  66220.390625   
 4 2021-01-04  15442.806641     11424.232422  74398.164062   
 
    avg_session_duration_sec  
 0                251.559174  
 1                262.139618  
 2                212.308212  
 3                188.636505  
 4                258.651245  ,
         date predicted_feature  predicted_value
 0 2020-12-31          sessions     19757.783203
 1 2021-01-01          sessions     14473.414062
 2 2021-01-02          sessions     14265.412109
 3 2021-01-03          sessions     14742.094727
 4 2021-01-04          sessions     15442.806641)

In [6]:
metrics_df_xgb

,target_feature,mae,rmse,mape,smape,r2
0,avg_session_duration_sec,56.858430,67.921707,31.992032,27.810159,-0.120443
1,unique_visitors,2023.024700,2663.923577,8.686142,8.963353,0.887543
2,sessions,2347.328975,3112.245218,7.664590,7.936209,0.908015
3,page_views,20195.739126,25516.798022,15.624131,15.831870,0.726816


In [7]:
import pandas as pd
from src.features import add_time_features
from src.models import SklearnRegressorConfig
from src.pipelines import fit_models_full

full_df = df.sort_values("date").reset_index(drop=True)

time_feature_cols_full = [
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "is_month_end",
    "is_month_start",
    "is_weekend",
]

traffic_feature_cols_full = [
    c
    for c in full_df.select_dtypes(include="number").columns
    if c not in time_feature_cols_full
]

targets_full = traffic_feature_cols_full

model_config_full = SklearnRegressorConfig(
    model_type="xgboost",
    random_state=42,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
)

models_full = fit_models_full(
    df=full_df,
    features=time_feature_cols_full,
    targets=targets_full,
    model_config=model_config_full,
    use_numpy=False,
)

future_test = pd.DataFrame({
    "date": pd.date_range("2023-01-01", "2024-07-01", freq="D")
})
future_test = add_time_features(future_test, date_col="date")
X_future_test = future_test[time_feature_cols_full]

pred_test_df = future_test[["date"]].copy()
for target, model in models_full.items():
    pred_test_df[target] = model.predict(X_future_test)

out_results_dir = project_root / "results"
out_results_dir.mkdir(parents=True, exist_ok=True)
out_results = out_results_dir / "webtraffic_predictions.csv"

pred_test_df.to_csv(out_results, index=False)

out_results, pred_test_df.head()

(WindowsPath('D:/MyML/datathon2026/results/webtraffic_predictions.csv'),
         date      sessions  unique_visitors    page_views  \
 0 2023-01-01  17071.623047     13097.431641  80610.343750   
 1 2023-01-02  16229.245117     11920.652344  82802.632812   
 2 2023-01-03  16299.270508     12368.006836  83281.007812   
 3 2023-01-04  16771.234375     12092.073242  86541.164062   
 4 2023-01-05  17100.017578     12677.872070  76037.968750   
 
    avg_session_duration_sec  
 0                261.117645  
 1                177.954376  
 2                208.668030  
 3                172.161713  
 4                189.636383  )